In [ ]:
import json
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

In [ ]:
# Path to dataset
DATA_RAW = Path("../data/raw")
DATA_SPLITS = Path("../data/splits")
DATA_SPLITS.mkdir(parents=True, exist_ok=True)

print(f"Dataset path: {DATA_RAW}")
print(f"Exists: {DATA_RAW.exists()}")

# 1. Remove duplicted and corrupted images

In [ ]:
hashes = {}
removed_corrupted = 0
removed_duplicates = 0

for img_path in DATA_RAW.rglob("*"):
    if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        try:
            # Check if the image is corrupted
            with Image.open(img_path) as img:
                img.verify()

            # Calculating the hash to detect duplicates
            with open(img_path, "rb") as f:
                file_hash = hashlib.md5(f.read()).hexdigest()

            if file_hash in hashes:
                print(f" Duplicate: {img_path}")
                img_path.unlink()
                removed_duplicates += 1
            else:
                hashes[file_hash] = img_path

        except Exception:
            print(f" Corrupted: {img_path}")
            img_path.unlink()
            removed_corrupted += 1

print(f"\n Removed corrupted images: {removed_corrupted}")
print(f" Removed duplicates: {removed_duplicates}")

# 2. Class distribution

In [ ]:
# Load classes (folders)
classes = sorted([d.name for d in DATA_RAW.iterdir() if d.is_dir()])
print(f"Classes trouvées ({len(classes)}) : {classes}")

# Count the images by class
class_counts = {}
all_images = []

for cls in classes:
    images = list((DATA_RAW / cls).glob("*.jpg")) + \
             list((DATA_RAW / cls).glob("*.jpeg")) + \
             list((DATA_RAW / cls).glob("*.png"))
    class_counts[cls] = len(images)
    all_images.extend([(str(img), cls) for img in images])

total = sum(class_counts.values())
print(f"\nTotal images : {total}")
print("\nPar classe :")
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    print(f"  {cls:<12} {count:>5} images ({pct:.1f}%)")

In [ ]:
# Distribution visualization
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#5DCAA5", "#378ADD", "#EF9F27", "#D85A30", "#7F77DD", "#888780",
          "#4CB944", "#F25F5C", "#9B5DE5", "#00A6A6", "#F76C6C"]
bars = ax.bar(class_counts.keys(), class_counts.values(), color=colors[:len(classes)], edgecolor="white", linewidth=0.5)

for bar, count in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(count), ha="center", va="bottom", fontsize=10)

ax.set_title("Distribution des classes", fontsize=13, pad=15)
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'images")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../data/splits/class_distribution.png", dpi=150)
plt.show()

# 3. Class Imbalance

In [ ]:
min_count = min(class_counts.values())
max_count = max(class_counts.values())
ratio = max_count / min_count

print(f"Classe la plus petite : {min(class_counts, key=class_counts.get)} ({min_count} images)")
print(f"Classe la plus grande : {max(class_counts, key=class_counts.get)} ({max_count} images)")
print(f"Ratio déséquilibre    : {ratio:.2f}")

if ratio > 3:
    print("\n ! Déséquilibre important — utiliser class_weight dans la loss")
elif ratio > 1.5:
    print("\n ! Déséquilibre modéré — surveiller les métriques par classe")
else:
    print("\n✓  Dataset équilibré")

# Calculating class weights for PyTorch
class_weights = {cls: total / (len(classes) * count) for cls, count in class_counts.items()}
print("\nClass weights suggérés :")
for cls, w in class_weights.items():
    print(f"  {cls:<12} {w:.3f}")

# 4. View images by class

In [ ]:
N_SAMPLES = 5
fig, axes = plt.subplots(len(classes), N_SAMPLES, figsize=(15, 3 * len(classes)))

for i, cls in enumerate(classes):
    images_paths = list((DATA_RAW / cls).glob("*.jpg")) + \
                   list((DATA_RAW / cls).glob("*.jpeg")) + \
                   list((DATA_RAW / cls).glob("*.png"))
    samples = np.random.choice(len(images_paths), min(N_SAMPLES, len(images_paths)), replace=False)

    for j, idx in enumerate(samples):
        img = Image.open(images_paths[idx]).convert("RGB")
        axes[i][j].imshow(img)
        axes[i][j].axis("off")
        if j == 0:
            axes[i][j].set_ylabel(cls, fontsize=11, rotation=0, labelpad=60, va="center")

plt.suptitle("Exemples par classe", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../data/splits/samples_per_class.png", dpi=100, bbox_inches="tight")
plt.show()

# 5. Image Dimensions and Sizes

In [ ]:
# Sample 200 images to analyze the dimensions
sample_paths = np.random.choice(len(all_images), min(200, len(all_images)), replace=False)
widths, heights = [], []

for idx in sample_paths:
    path, _ = all_images[idx]
    with Image.open(path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

print("Largeur  — min:", min(widths), "max:", max(widths), "moyenne:", int(np.mean(widths)))
print("Hauteur  — min:", min(heights), "max:", max(heights), "moyenne:", int(np.mean(heights)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color="#5DCAA5", edgecolor="white")
axes[0].set_title("Distribution largeurs")
axes[0].axvline(224, color="#D85A30", linestyle="--", label="resize target 224px")
axes[0].legend()
axes[1].hist(heights, bins=30, color="#378ADD", edgecolor="white")
axes[1].set_title("Distribution hauteurs")
axes[1].axvline(224, color="#D85A30", linestyle="--", label="resize target 224px")
axes[1].legend()
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../data/splits/image_dimensions.png", dpi=150)
plt.show()

print("\n→ Resize recommandé : 224x224")

# 6. RGB Distribution and Mean/Standard Deviation Calculation

In [ ]:
# Calculate the mean and standard deviation for 300 images (required for normalization)
TARGET_SIZE = (224, 224)
sample_paths = np.random.choice(len(all_images), min(300, len(all_images)), replace=False)

pixels_r, pixels_g, pixels_b = [], [], []

for idx in sample_paths:
    path, _ = all_images[idx]
    img = Image.open(path).convert("RGB").resize(TARGET_SIZE)
    arr = np.array(img) / 255.0
    pixels_r.extend(arr[:, :, 0].flatten())
    pixels_g.extend(arr[:, :, 1].flatten())
    pixels_b.extend(arr[:, :, 2].flatten())

mean = [np.mean(pixels_r), np.mean(pixels_g), np.mean(pixels_b)]
std  = [np.std(pixels_r),  np.std(pixels_g),  np.std(pixels_b)]

print("Mean par canal (R, G, B) :", [round(m, 4) for m in mean])
print("Std  par canal (R, G, B) :", [round(s, 4) for s in std])
print("\n À utiliser dans preprocessing :")
print(f"  MEAN = {[round(m, 4) for m in mean]}")
print(f"  STD  = {[round(s, 4) for s in std]}")

# RGB Distribution Visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_rgb = [("#E24B4A", "R"), ("#639922", "G"), ("#378ADD", "B")]
for ax, pixels, (color, label) in zip(axes, [pixels_r, pixels_g, pixels_b], colors_rgb):
    ax.hist(pixels, bins=50, color=color, alpha=0.7, edgecolor="none")
    ax.set_title(f"Canal {label}")
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Distribution des valeurs RGB (normalisées 0-1)", fontsize=13)
plt.tight_layout()
plt.savefig("../data/splits/rgb_distribution.png", dpi=150)
plt.show()

# 7. Mapping labels

In [ ]:
# Manual mapping : classe → matière + recyclable
# Source of truth for the ‘classes’ table in PostgreSQL
LABEL_MAP = {
    "battery": {
        "type": "Pile/Batterie",
        "recyclable": True,
        "bac": "Point de collecte",
        "alt": "Supermarché ou magasin",
        "advice": "Dépose-les dans un point de collecte, ne les jette jamais à la poubelle"
    },
    "cardboard": {
        "type": "Carton",
        "recyclable": True,
        "bac": "Bac jaune",
        "alt": None,
        "advice": "À plier pour gagner de la place"
    },
    "electronic": {
        "type": "Électronique",
        "recyclable": True,
        "bac": "Déchèterie",
        "alt": "Magasin (reprise gratuite)",
        "advice": "Le magasin reprend ton ancien appareil lors d’un achat, et accepte aussi les petits appareils sans obligation d’achat"
    },
    "glass": {
        "type": "Verre",
        "recyclable": True,
        "bac": "Bac blanc",
        "alt": "Borne à verre (verte)",
        "advice": "Pense à enlever les bouchons et capsules avant de jeter"
    },
    "metal": {
        "type": "Métal",
        "recyclable": True,
        "bac": "Bac jaune",
        "alt": "Déchèterie (pour les gros objets métalliques)",
        "advice": "Boîtes de conserve, canettes acceptées"
    },
    "organic": {
        "type": "Déchets organiques",
        "recyclable": False,
        "bac": "Bac marron",
        "alt": "Compost maison",
        "advice": "Le tri des biodéchets est désormais obligatoire (depuis 2024), si une solution est disponible près de chez toi"
    },
    "paper": {
        "type": "Papier",
        "recyclable": True,
        "bac": "Bac jaune",
        "alt": "Point tri",
        "advice": "Pas de papier gras ou mouillé"
    },
    "plastic": {
        "type": "Plastique",
        "recyclable": True,
        "bac": "Bac jaune",
        "alt": "Déchèterie (pour les plastiques durs ou volumineux)",
        "advice": "Tous les emballages plastiques sont maintenant acceptés grâce à l'extension des consignes de tri"
    },
    "textile": {
        "type": "Textile",
        "recyclable": True,
        "bac": "Borne textile",
        "alt": "Associations (Emmaüs, Croix-Rouge)",
        "advice": "Même usés, à déposer propres et secs dans un sac fermé"
    },
    "medical": {
        "type": "Déchets médicaux",
        "recyclable": True,
        "bac": "Borne textile",
        "alt": "Retour à un point de collecte médical",
        "advice": (
            "Les déchets médicaux tels que seringues, aiguilles, stylos injecteurs et pansements "
            "doivent être déposés dans des points de collecte spécialisés. Ne pas les jeter dans la poubelle classique. "
            "Si l'emballage est vide, il peut être jeté dans la poubelle appropriée (comme le plastique ou le papier), "
            "mais uniquement après avoir correctement éliminé le contenu et s'assurer qu'il ne présente aucun danger."
        )
    },
    "trash": {
        "type": "Déchets",
        "recyclable": False,
        "bac": "Bac gris",
        "alt": "Déchèterie",
        "advice": "Utilise cette poubelle uniquement si tu ne peux pas trier autrement"
    },
}

# Add the numerical index (required for the CNN)
for i, cls in enumerate(sorted(LABEL_MAP.keys())):
    LABEL_MAP[cls]["index"] = i

print("Mapping classes :")
for cls, info in LABEL_MAP.items():
    recyclable = "Recyclable" if info["recyclable"] else "Non recyclable"
    print(f"  {cls:<12} → {info['type']:<18} {recyclable}")

# Save as JSON
with open("../data/splits/label_map.json", "w", encoding="utf-8") as f:
    json.dump(LABEL_MAP, f, ensure_ascii=False, indent=2)

print("\n label_map.json sauvegardé dans model/data/splits/")

## 8. Split train / val / test

In [ ]:
# Split stratify 80 / 10 / 10
paths = [p for p, _ in all_images]
labels = [label for _, label in all_images]

# Train 80% / val and test 20%
X_train, X_temp, y_train, y_temp = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=42
)
# Val 10% / Test 10%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train : {len(X_train)} images")
print(f"Val   : {len(X_val)} images")
print(f"Test  : {len(X_test)} images")
print(f"Total : {len(X_train) + len(X_val) + len(X_test)} images")

# Check that the split is properly layered
print("\nDistribution train :")
for cls, count in sorted(Counter(y_train).items()):
    print(f"  {cls:<12} {count}")

In [ ]:
# Save splits as JSON
splits = {
    "train": [{"path": p, "label": label} for p, label in zip(X_train, y_train)],
    "val":   [{"path": p, "label": label} for p, label in zip(X_val, y_val)],
    "test":  [{"path": p, "label": label} for p, label in zip(X_test, y_test)],
}

with open("../data/splits/splits.json", "w") as f:
    json.dump(splits, f, indent=2)

# Save the dataset statistics
stats = {
    "total_images": total,
    "classes": classes,
    "class_counts": class_counts,
    "class_weights": class_weights,
    "imbalance_ratio": round(ratio, 2),
    "image_size_target": [224, 224],
    "mean_rgb": [round(m, 4) for m in mean],
    "std_rgb":  [round(s, 4) for s in std],
    "splits": {
        "train": len(X_train),
        "val":   len(X_val),
        "test":  len(X_test),
    }
}

with open("../data/splits/dataset_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("splits.json sauvegardé")
print("dataset_stats.json sauvegardé")
print("\nFichiers générés dans model/data/splits/ :")
for f in sorted(DATA_SPLITS.iterdir()):
    print(f"  {f.name}")